In [2]:
include("../RayTracing.jl")

Main.RayTracing

In [3]:
using BenchmarkTools

In [4]:
parsed_args = RayTracing.parse_commandline()
parsed_args["scene-number"] = 4
    
# set up logging
logger = RayTracing.setup_logging(parsed_args["debug"])
RayTracing.global_logger(logger)

# set random seed
RayTracing.Random.seed!(parsed_args["seed"])

Random.TaskLocalRNG()

In [5]:
I, scene = RayTracing.build_scene(parsed_args)


There are 37 objects in the scene, building BVH
  0.024397 seconds (78.71 k allocations: 5.577 MiB, 98.67% compilation time)
Done building BVH
Using 5 samples per pixel
There are 2 lights in the scene


(Main.RayTracing.BDPTIntegrator(Main.RayTracing.PerspectiveCamera(Main.RayTracing.ProjectiveCamera(Main.RayTracing.CameraCore(Main.RayTracing.Transformation([1.0 0.0 0.0 278.0; 0.0 1.0 0.0 278.0; 0.0 0.0 1.0 -800.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 -278.0; 0.0 1.0 0.0 -278.0; 0.0 0.0 1.0 800.0; 0.0 0.0 0.0 1.0]), 0.0, 1.0, Main.RayTracing.Film([250.0, 250.0], Main.RayTracing.Bounds2([0.0, 0.0], [250.0, 250.0]), 0.001, Main.RayTracing.BoxFilter([0.1, 0.1]), "yeehaw.exr", Main.RayTracing.Pixel[Main.RayTracing.Pixel([0.0, 0.0, 0.0], 0.0, Main.RayTracing.AtomicXYZPBRT(Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0))) Main.RayTracing.Pixel([0.0, 0.0, 0.0], 0.0, Main.RayTracing.AtomicXYZPBRT(Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0))) … Main.RayTracing.Pixel([0.0, 0.0, 0.0], 0.0, Main.RayTracing.AtomicXYZPBRT(Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0),

# Descending into `render()`

In [7]:
parsed_args["light-distribution-strategy"]

"uniform"

In [32]:
@btime light_distr_generator = RayTracing.LightDistribution(
    parsed_args["light-distribution-strategy"], 
    scene,
    parsed_args["weight-for-infinites"], 
    parsed_args["n-voxels"], 
    parsed_args["n-shadow-rays"], 
)
light_distr_generator = RayTracing.LightDistribution(
    parsed_args["light-distribution-strategy"], 
    scene,
    parsed_args["weight-for-infinites"], 
    parsed_args["n-voxels"], 
    parsed_args["n-shadow-rays"], 
)

  226.812 ns (3 allocations: 192 bytes)


Main.RayTracing.StaticLightDistribution(Main.RayTracing.Distribution1D([1.0, 1.0], [0.0, 0.5, 1.0], 1.0))

In [10]:
sample_bounds = RayTracing.get_sample_bounds(I.camera.core.core.film)
sample_extent = RayTracing.diagonal(sample_bounds)
tile_size = 16
width, height = Int64.(floor.((sample_extent .+ tile_size) ./ tile_size))
total_tiles = width * height - 1

255

In [11]:
k = 35

35

In [20]:
x, y = k % width, k ÷ width
tile = RayTracing.Pnt2(x, y)
@btime sampler = RayTracing.deepcopy(I.sampler)
sampler = RayTracing.deepcopy(I.sampler)

  2.935 μs (4 allocations: 432 bytes)


Main.RayTracing.ZSobolSampler(2, 5, 0, 9, 2, 0x0000000000000000, 0)

In [15]:
tb_min = sample_bounds.pMin .+ tile .* tile_size
tb_max = min.(tb_min .+ (tile_size - 1), sample_bounds.pMax)
tile_bounds = RayTracing.Bounds2(tb_min, tb_max)
@btime film_tile = RayTracing.FilmTile(I.camera.core.core.film, tile_bounds)

  2.231 μs (280 allocations: 15.58 KiB)


Main.RayTracing.FilmTile(Main.RayTracing.Bounds2([48.0, 32.0], [63.0, 47.0]), [0.1, 0.1], [10.0, 10.0], [1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0; … ; 1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0], 16, Main.RayTracing.FilmTilePixel[Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0) Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0) … Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0) Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0); Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0) Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0) … Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0) Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0); … ; Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0) Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0) … Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0) Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0); Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0) Main.RayTracing.FilmTilePixel([0.0, 0.0, 0.0], 0.0) … Main.RayTracing.FilmTilePixel([0.0, 0

In [17]:
pixel = RayTracing.Pnt2(54, 33)

2-element Main.RayTracing.Pnt2 with indices SOneTo(2):
 54.0
 33.0

In [22]:
sample_index = 2

2

In [23]:
RayTracing.start_pixel_sample!(sampler, pixel, sample_index-1)

0x0000000000003459

In [24]:
camera_sample = RayTracing.get_camera_sample!(sampler, pixel)

Main.RayTracing.CameraSample([54.29972175206058, 33.464278222527355], [0.5121479292865843, 0.2182439980097115], 0.12401669309474528)

In [25]:
L = RayTracing.spectrum_from_float(0.0)
# instantiate the list of vertices
camera_vertices = Vector{RayTracing.Vertex}(undef, I.max_depth + 2)
light_vertices = Vector{RayTracing.Vertex}(undef, I.max_depth + 1)

7-element Vector{Main.RayTracing.Vertex}:
 #undef
 #undef
 #undef
 #undef
 #undef
 #undef
 #undef

In [27]:
@btime RayTracing.generate_camera_subpath!(
    camera_vertices,
    scene, 
    sampler, 
    I.max_depth + 2, 
    I.camera,
    camera_sample, 
)

  16.375 μs (218 allocations: 14.27 KiB)


8

In [28]:
n_camera = RayTracing.generate_camera_subpath!(
    camera_vertices,
    scene, 
    sampler, 
    I.max_depth + 2, 
    I.camera,
    camera_sample, 
)

3

In [33]:
light_distr = RayTracing.lookup(light_distr_generator, RayTracing.p(camera_vertices[1]))

Main.RayTracing.Distribution1D([1.0, 1.0], [0.0, 0.5, 1.0], 1.0)

In [34]:
@btime RayTracing.generate_light_subpath!(
    light_vertices,
    scene,
    sampler,
    I.max_depth + 1,
    RayTracing.time(camera_vertices[1]),
    light_distr
)

  8.625 μs (155 allocations: 8.80 KiB)


(7, 2)

In [37]:
light_distr = RayTracing.lookup(light_distr_generator, RayTracing.p(camera_vertices[1]))
n_light, light_num = RayTracing.generate_light_subpath!(
    light_vertices,
    scene,
    sampler,
    I.max_depth + 1,
    RayTracing.time(camera_vertices[1]),
    light_distr
)

(4, 1)

In [40]:
bdpt_pass = (-1,-1)
for t in 1:n_camera
    for s in 0:n_light
        if ((s,t) == bdpt_pass) || (bdpt_pass == (-1,-1))
            depth = t + s - 2
            if ((s==1)&&(t==1) || (depth<0) || (depth>I.max_depth))
                continue
            end


            println("$t, $s")
        end
    end
end

1, 2
1, 3
1, 4
2, 0
2, 1
2, 2
2, 3
2, 4
3, 0
3, 1
3, 2
3, 3
3, 4


In [45]:
mis_weight = 0.0
@btime RayTracing.connect_BDPT(
    scene,
    light_vertices,
    camera_vertices,
    1,
    4,
    light_distr,
    light_num,
    I.camera,
    sampler,
    camera_sample.film
)


  5.375 μs (88 allocations: 4.50 KiB)


([0.0, 0.0, 0.0], 0.0, [54.29972175206058, 33.464278222527355])